In [ ]:
%pip install pypdf==7.1.1

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pathlib import Path
from pypdf import PdfReader
from pyspark.sql import functions as F
import hashlib

RAW_DIR = Path("/Volumes/workspace/viktoriia_kalenichenko_lab12/raw_docs")
TABLE = "workspace.viktoriia_kalenichenko_lab12.document_pages"
FILES = ["P510294-1b00a8b1-c76f-419d-953a-46bd5805c044.pdf"]

for filename in FILES:
    path = RAW_DIR / filename

    if not path.is_file(): raise FileNotFoundError(f"Not found: {path}")

    with path.open("rb") as stream:
        header = stream.read(1024)

    if b"%PDF-" not in header:raise ValueError(f"The file does not have the expected PDF header: {filename}")

    print(f"OK | {filename} | {path.stat().st_size:,} bytes")

In [ ]:
rows = []

for filename in FILES:
    path = RAW_DIR / filename
    file_hash = hashlib.sha256(path.read_bytes()).hexdigest()

    with path.open("rb") as stream:
        reader = PdfReader(stream)

        if reader.is_encrypted:
            raise ValueError(f"PDF is encrypted: {filename}")

        page_count = len(reader.pages)
        if page_count == 0:
            raise ValueError(f"PDF contains no pages: {filename}")

        for page_number, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text() or ""
            except Exception as exc:
                raise RuntimeError(
                    f"Error: {filename}, page {page_number}"
                ) from exc

            rows.append((
                filename,
                str(path),
                file_hash,
                page_number,
                text,
                len(text.strip()),
            ))

    print(f"{filename}: {page_count} pages")

pages_df = spark.createDataFrame(
    rows,
    schema="""
        file_name STRING,
        file_path STRING,
        source_sha256 STRING,
        page_number INT,
        text STRING,
        char_count INT
    """,
)

print(f"Total pages retrieved: {len(rows)}")

In [ ]:
from pyspark.sql import functions as F

display(
    pages_df.groupBy("file_name").agg(
        F.count("*").alias("total_pages"),
        F.sum(
            F.when(F.col("char_count") == 0, 1).otherwise(0)
        ).alias("empty_pages"),
        F.sum(
            F.when(F.col("char_count") < 300, 1).otherwise(0)
        ).alias("pages_under_300_chars"),
        F.round(F.avg("char_count"), 0).alias("avg_chars_per_page"),
    )
)

In [ ]:
print("КОРОТКІ СТОРІНКИ")

display(
    pages_df
    .filter(F.col("char_count") < 300)
    .select("page_number", "char_count", "text")
    .orderBy("page_number")
)

print("ПРИКЛАДИ ОСНОВНОГО ТЕКСТУ")

samples = (
    pages_df
    .filter(F.col("page_number").isin(20, 60, 100))
    .select("page_number", "text")
    .orderBy("page_number")
    .collect()
)

for row in samples:
    print(f"\n===== PDF-сторінка {row['page_number']} =====\n")
    print(row["text"][:2500])

In [ ]:
samples = (
    pages_df
    .filter(F.col("page_number").isin(20, 60, 100))
    .select("page_number", "text")
    .orderBy("page_number")
    .collect()
)

for row in samples:
    print(f"\n===== PDF-сторінка {row['page_number']} =====\n")
    print(row["text"][:2500])

In [ ]:
TABLE = "workspace.viktoriia_kalenichenko_lab12.document_pages"

# Перевіряємо результат перед записом.
assert pages_df.count() == 147, "Очікували 147 сторінок"
assert pages_df.select("file_name").distinct().count() == 1, \
    "Очікували один документ"

assert (
    pages_df.groupBy("file_name", "page_number")
    .count()
    .filter(F.col("count") > 1)
    .count() == 0
), "Знайдено дублікати сторінок"

# Зберігаємо початковий текст у Delta.
pages_df.write.format("delta").mode("overwrite").saveAsTable(TABLE)

# Читаємо вже збережену таблицю.
saved_df = spark.table(TABLE)
saved_count = saved_df.count()

assert saved_count == 147, "Кількість збережених сторінок не збігається"
print(f"Збережено {saved_count} сторінок у {TABLE}")